# Notebook 02 — ASR data preflight & engineering smoke test

**Mục tiêu:** khóa input từ Notebook 01, chuẩn hóa text Bahnar + CTC vocab, QA waveform **có chọn mẫu** (không tải 74.8 GB), cache audio nhỏ, chạy ASR smoke test kỹ thuật.

**Không làm:** chia lại split, sửa manifest RQ1, fine-tune full ASR, WER/CER nghiên cứu, đụng frozen test cho vocab/model, RQ2/S2TT/MT.

> The frozen test set remains sealed in Notebook 02. It is not used for vocabulary construction, model selection, smoke training, or preliminary evaluation.


## Mapping với kế hoạch

| Việc | Vì sao | Output |
|------|--------|--------|
| Khóa manifest + SHA | Reproducibility RQ1 | `notebook02_manifest_checks.csv` |
| CTC vocab từ train | Tokenizer cho Notebook 03 | `artifacts/tokenizers/bahnar_char_ctc/` |
| Sampled audio QA | Preflight kỹ thuật, **không** claim đã QA 590 giờ | `notebook02_*_audio_qa*.csv` |
| Smoke ASR | Pipeline không crash | `results/notebook02_smoke_test.json` |

**Gate:** chỉ in `NOTEBOOK 02 COMPLETED — READY FOR NOTEBOOK 03` khi mọi mandatory check pass.


In [1]:
# Cell 1 — Bootstrap (Mac local ưu tiên; Colab tương thích)
from pathlib import Path
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

IN_COLAB = "google.colab" in sys.modules

def _is_root(p: Path) -> bool:
    return (
        p.is_dir()
        and (p / "requirements.txt").is_file()
        and (p / "src").is_dir()
        and (p / "data" / "manifests").is_dir()
    )

def resolve_project_root() -> Path:
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
        for cand in (
            Path("/content/drive/MyDrive/bahnar-s2tt-thesis"),
            Path("/content/bahnar-s2tt-thesis"),
        ):
            if _is_root(cand):
                return cand.resolve()
    cwd = Path.cwd().resolve()
    for cand in [cwd, cwd.parent, *cwd.parents]:
        if _is_root(cand):
            return cand.resolve()
    raise FileNotFoundError(
        "Cannot locate project root (needs requirements.txt, src/, data/manifests/)."
    )

PROJECT_ROOT = resolve_project_root()
REQ = PROJECT_ROOT / "requirements.txt"
print("PROJECT_ROOT:", PROJECT_ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ)])
print("Dependencies installed.")


PROJECT_ROOT: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis
Dependencies installed.


In [2]:
# Cell 2 — Imports, paths, device, seed, sessions + PYTEST GATE
from __future__ import annotations
import hashlib
import json
import os
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import importlib
import requests
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def _is_root(p: Path) -> bool:
    return (
        Path(p).is_dir()
        and (Path(p) / "requirements.txt").is_file()
        and (Path(p) / "src").is_dir()
        and (Path(p) / "data" / "manifests").is_dir()
    )

if "PROJECT_ROOT" in globals() and _is_root(Path(PROJECT_ROOT)):
    PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = next((c for c in [cwd, cwd.parent, *cwd.parents] if _is_root(c)), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("PROJECT_ROOT not found; run Cell 1 first.")
    PROJECT_ROOT = Path(PROJECT_ROOT).resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.audio_utils as audio_utils
importlib.reload(audio_utils)

import src.data_utils as data_utils
importlib.reload(data_utils)

from src.audio_utils import check_waveform, waveform_to_mono_float32
from src.data_utils import (
    AUDIO_PROCESSING_VERSION,
    NORMALIZATION_VERSION,
    assert_no_signed_urls_persisted,
    asset_headers,
    audit_train_script_contamination,
    build_cache_provenance,
    build_char_ctc_vocab,
    cache_provenance_matches,
    character_frequency,
    check_manifest_contract,
    classify_character,
    compute_contamination_summary,
    dataset_viewer_get,
    download_audio_bytes,
    duration_bin,
    filter_candidates_by_duration,
    NOTEBOOK03_COMPAT_MIN_DURATION,
    NOTEBOOK03_COMPAT_MAX_DURATION,
    encode_text_with_vocab,
    extract_audio_src,
    fetch_hub_dataset_sha,
    find_oov_characters,
    find_oov_rows,
    generate_run_id,
    is_valid_sha256,
    load_cache_sidecar,
    load_frozen_test_restricted,
    load_manifest,
    normalize_bahnar_ctc_v1,
    parse_bool,
    parse_bool_series,
    process_audio_candidate,
    project_paths,
    read_wav,
    redact_url,
    resample_audio,
    run_metadata,
    safe_cache_filename,
    sample_distribution_report,
    save_tokenizer_clean,
    save_vocab_artifacts,
    select_representative_samples,
    sha256_file,
    strip_signed_url_columns,
    texts_match_after_light_norm,
    to_mono_float32,
    uniform_window_offsets,
    validate_cache_wav,
    validate_https_url,
    verify_json_metadata,
    verify_report_metadata,
    viewer_headers,
    write_cache_sidecar,
    write_dataframe_csv,
    write_pcm16_wav,
)

PATHS = project_paths(PROJECT_ROOT)
for key in ("audit", "cache_audio", "results"):
    PATHS[key].mkdir(parents=True, exist_ok=True)

# Reusable HTTP sessions
VIEWER_SESSION = requests.Session()
ASSET_SESSION = requests.Session()

import torch
torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = "cuda"
    torch.cuda.manual_seed_all(SEED)
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
    try:
        torch.mps.manual_seed(SEED)
    except Exception:
        pass
else:
    DEVICE = "cpu"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
print("Normalization:", NORMALIZATION_VERSION)
print("Audio processing:", AUDIO_PROCESSING_VERSION)

# ==========================================================================
# MANDATORY PYTEST GATE
# ==========================================================================
print("\n" + "="*60)
print("RUNNING PYTEST GATE")
print("="*60)

pytest_cmd = [
    sys.executable, "-m", "pytest", "-q",
    "tests/test_audio_utils.py",
    "tests/test_data_utils.py",
]

# Check if test files exist
test_files_exist = all((PROJECT_ROOT / f).exists() for f in ["tests/test_audio_utils.py", "tests/test_data_utils.py"])
if not test_files_exist:
    print("WARNING: Some test files missing, running available tests only")
    pytest_cmd = [sys.executable, "-m", "pytest", "-q", "tests/"]

pytest_result = subprocess.run(
    pytest_cmd,
    cwd=str(PROJECT_ROOT),
    capture_output=True,
    text=True,
)

pytest_stdout = pytest_result.stdout
pytest_stderr = pytest_result.stderr
pytest_returncode = pytest_result.returncode

# Save pytest output
pytest_txt_path = PATHS["results"] / "notebook02_pytest.txt"
pytest_txt_path.write_text(f"COMMAND: {' '.join(pytest_cmd)}\n\nSTDOUT:\n{pytest_stdout}\n\nSTDERR:\n{pytest_stderr}\n\nRETURN CODE: {pytest_returncode}\n", encoding="utf-8")

# Parse test summary
pytest_passed = pytest_returncode == 0
pytest_summary = "PASS" if pytest_passed else "FAIL"

# Extract test counts from output (e.g., "61 passed in 2.43s")
import re
match = re.search(r"(\d+)\s+passed", pytest_stdout)
tests_passed = int(match.group(1)) if match else 0
match_failed = re.search(r"(\d+)\s+failed", pytest_stdout)
tests_failed = int(match_failed.group(1)) if match_failed else 0

print(pytest_stdout)
if pytest_stderr:
    print("STDERR:", pytest_stderr)

print(f"\nPYTEST SUMMARY: {pytest_summary}")
print(f"Tests passed: {tests_passed}, failed: {tests_failed}")

if not pytest_passed:
    raise RuntimeError(f"PYTEST GATE FAILED — {tests_failed} tests failed. Fix tests before continuing.")

print("PYTEST GATE: PASS")


/Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


PROJECT_ROOT: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis
DEVICE: mps
Normalization: bahnar_ctc_normalization_v1
Audio processing: notebook02_audio_v3

RUNNING PYTEST GATE
........................................................................ [ 52%]
..................................................................       [100%]
=============================== warnings summary ===============================
tests/test_data_utils.py::test_download_audio_does_not_forward_authorization
  /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
138 passed, 1 warning in 2.33s


PYTEST SU

## Cấu hình thí nghiệm (engineering)

Các ngưỡng dưới đây là **preflight / smoke**, không phải hyperparameter nghiên cứu RQ1.


In [3]:
# Cell 3 — Configuration (locked to Notebook 01 contract)
DATASET_ID = "cuong06/Bahnar_Vietnamese"
CONFIG_NAME = "default"
EXPECTED_DATASET_REVISION = "3d88d3951b1a6e3388559b341cd7bd274879d696"

EXPECTED_COUNTS = {
    "train": 102_698,
    "validation": 11_132,
    "test": 215,
}

# SHA256 from split_summary.json (authoritative)
split_summary = json.loads((PATHS["manifests"] / "split_summary.json").read_text(encoding="utf-8"))
EXPECTED_MANIFEST_SHA256 = dict(split_summary.get("manifest_sha256") or {})
for fname, digest in EXPECTED_MANIFEST_SHA256.items():
    if not is_valid_sha256(digest):
        raise RuntimeError(f"Invalid SHA256 in split_summary for {fname}: {digest!r}")

# Sampling parameters
QA_TRAIN_TARGET = 500
QA_VALIDATION_TARGET = 70
VIEWER_WINDOW_LENGTH = 100
VIEWER_MAX_WINDOWS = 25
TARGET_SAMPLING_RATE = 16_000
AUDIO_MIN_DURATION = 0.05
AUDIO_MAX_DURATION = 120.0
# Filter candidates BEFORE sampling/cache so Notebook 03 (0.5–30s) can use them.
CACHE_SAMPLE_MIN_DURATION = NOTEBOOK03_COMPAT_MIN_DURATION  # 0.5
CACHE_SAMPLE_MAX_DURATION = NOTEBOOK03_COMPAT_MAX_DURATION  # 30.0
HARD_OK_RATE_MIN = 0.95

# Smoke test config - PINNED REVISION
SMOKE_MODEL_ID = "hf-internal-testing/tiny-random-wav2vec2"
SMOKE_MODEL_REVISION = "9123c4c809823cc53466e9868a1cf1c476be2e54"
SMOKE_TRAIN_EXAMPLES = 4
SMOKE_VALIDATION_EXAMPLES = 2
SMOKE_TRAIN_STEPS = 2

# Generate unique run ID for this execution
RUN_ID = generate_run_id()

# Auth headers (VIEWER may have token, ASSET never does)
HF_TOKEN = os.environ.get("HF_TOKEN")
VIEWER_HEADERS = viewer_headers(HF_TOKEN)
ASSET_HEADERS = asset_headers()
assert "Authorization" not in ASSET_HEADERS, "ASSET_HEADERS must not have Authorization"

if HF_TOKEN:
    print("HF authentication: enabled (VIEWER only)")
else:
    print("HF authentication: anonymous")

RUN_META = run_metadata(
    run_id=RUN_ID,
    dataset_id=DATASET_ID,
    dataset_revision=EXPECTED_DATASET_REVISION,
    seed=SEED,
    processing_version=AUDIO_PROCESSING_VERSION,
    input_sha256=EXPECTED_MANIFEST_SHA256,
)

# Save pytest report JSON with run metadata
pytest_json_path = PATHS["results"] / "notebook02_pytest.json"
pytest_json_path.write_text(json.dumps({
    **RUN_META,
    "command": pytest_cmd,
    "return_code": pytest_returncode,
    "tests_passed": tests_passed,
    "tests_failed": tests_failed,
    "pytest_passed": pytest_passed,
    "summary": pytest_summary,
}, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"RUN_ID: {RUN_ID}")
print(json.dumps({
    "dataset": DATASET_ID,
    "expected_revision": EXPECTED_DATASET_REVISION,
    "counts": EXPECTED_COUNTS,
    "qa_targets": {"train": QA_TRAIN_TARGET, "validation": QA_VALIDATION_TARGET},
    "viewer_window_length": VIEWER_WINDOW_LENGTH,
    "viewer_max_windows": VIEWER_MAX_WINDOWS,
    "smoke_model": SMOKE_MODEL_ID,
    "smoke_model_revision": SMOKE_MODEL_REVISION,
    "device": DEVICE,
    "pytest_passed": pytest_passed,
}, indent=2))


HF authentication: anonymous
RUN_ID: d0478702-32f0-4bf6-ae1b-6932c190a16b
{
  "dataset": "cuong06/Bahnar_Vietnamese",
  "expected_revision": "3d88d3951b1a6e3388559b341cd7bd274879d696",
  "counts": {
    "train": 102698,
    "validation": 11132,
    "test": 215
  },
  "qa_targets": {
    "train": 500,
    "validation": 70
  },
  "viewer_window_length": 100,
  "viewer_max_windows": 25,
  "smoke_model": "hf-internal-testing/tiny-random-wav2vec2",
  "smoke_model_revision": "9123c4c809823cc53466e9868a1cf1c476be2e54",
  "device": "mps",
  "pytest_passed": true
}


## Step 1 — Khóa contract Manifest từ Notebook 01

Đọc CSV/JSON đã freeze; so SHA256 với `split_summary.json`; không sửa manifest.

**Dừng nếu fail** — không tự “fix” data.


In [4]:
# Cell 4 — Manifest contract + Hub SHA lock + frozen-test seal audit
HUB_SHA_BEFORE = fetch_hub_dataset_sha(DATASET_ID, token=HF_TOKEN)
print("Hub SHA (before):", HUB_SHA_BEFORE)

manifest_checks = check_manifest_contract(
    root=PROJECT_ROOT,
    dataset_id=DATASET_ID,
    expected_revision=EXPECTED_DATASET_REVISION,
    expected_counts=EXPECTED_COUNTS,
    hub_sha=HUB_SHA_BEFORE,
    split_summary=split_summary,
)
write_dataframe_csv(manifest_checks, PATHS["audit"] / "notebook02_manifest_checks.csv", metadata=RUN_META)
display(manifest_checks)

failed = manifest_checks.loc[~manifest_checks["passed"].map(parse_bool)]
if len(failed):
    raise RuntimeError(f"Manifest contract FAILED:\n{failed.to_string(index=False)}")

# Load manifests (train/val full, test restricted)
train_df = load_manifest("train", PROJECT_ROOT)
val_df = load_manifest("validation", PROJECT_ROOT)
test_df, frozen_seal_report = load_frozen_test_restricted(PROJECT_ROOT)

# Save frozen test seal report
frozen_seal_path = PATHS["audit"] / "notebook02_frozen_test_seal.json"
frozen_seal_path.write_text(json.dumps({**RUN_META, **frozen_seal_report}, ensure_ascii=False, indent=2), encoding="utf-8")

if not frozen_seal_report["passed"]:
    raise RuntimeError(f"Frozen test seal FAILED: loaded forbidden columns {frozen_seal_report['forbidden_columns_loaded']}")

print(f"Loaded manifests: train={len(train_df):,} validation={len(val_df):,} test={len(test_df):,} (restricted columns)")
print(f"Frozen test loaded columns: {frozen_seal_report['loaded_columns']}")
print(f"Frozen test forbidden columns in file (NOT loaded): {frozen_seal_report['forbidden_columns_in_file']}")
print("Frozen test SEALED in Notebook 02.")


Hub SHA (before): 3d88d3951b1a6e3388559b341cd7bd274879d696


,check,passed,detail
0,split_summary_exists,True,/Users/minhtuan25/Desktop/MinhTuanCode/MasterA...
1,dataset_id_match,True,cuong06/Bahnar_Vietnamese
2,dataset_revision_match,True,3d88d3951b1a6e3388559b341cd7bd274879d696
3,hub_sha_matches_locked,True,3d88d3951b1a6e3388559b341cd7bd274879d696
4,rq1_train.csv_exists,True,/Users/minhtuan25/Desktop/MinhTuanCode/MasterA...
5,rq1_train.csv_sha_valid_format,True,b9acbc3ff31411f22db0806b9cb1d59b518141ba7c7055...
6,rq1_train.csv_sha_match,True,b9acbc3ff31411f22db0806b9cb1d59b518141ba7c7055...
7,train_row_count,True,actual=102698 expected=102698
8,train_required_columns,True,
9,train_record_uid_unique,True,


Loaded manifests: train=102,698 validation=11,132 test=215 (restricted columns)
Frozen test loaded columns: ['record_uid', 'record_id', 'source_split', 'recording_group_id', 'group_id', 'pair_key', 'split']
Frozen test forbidden columns in file (NOT loaded): ['audio_path', 'duration_seconds', 'text_bahnar', 'text_vi', 'text_en', 'qa_ok', 'qa_hard_ok', 'qa_reason', 'qa_quality_warnings', 'qa_duration_sec']
Frozen test SEALED in Notebook 02.


## Step 2 — Chuẩn hóa text Bahnar & CTC character vocabulary

Vocab **chỉ** từ `rq1_train`. OOV chỉ audit trên validation (không đụng test).

Đây vừa là engineering check, vừa là bằng chứng phương pháp luận (tokenization CTC có kiểm soát).


In [5]:
# Cell 5 — Text normalization + contamination audit + clean splits + tokenizer
from src.data_utils import (
    audit_dataframe_contamination,
    build_exclusion_list,
    build_clean_split,
    verify_clean_split_no_contamination,
    compute_ordered_uid_hash,
    compute_uid_set_hash,
    export_clean_split_contract,
    verify_clean_split_contract,
    verify_tokenizer_provenance,
)

train_df["text_bahnar_norm"] = train_df["text_bahnar"].map(normalize_bahnar_ctc_v1)
val_df["text_bahnar_norm"] = val_df["text_bahnar"].map(normalize_bahnar_ctc_v1)

empty_train = int((train_df["text_bahnar_norm"].str.len() == 0).sum())
empty_val = int((val_df["text_bahnar_norm"].str.len() == 0).sum())
print(f"Empty normalized train texts: {empty_train}")
print(f"Empty normalized validation texts: {empty_val}")
if empty_train > 0:
    raise RuntimeError("Train has empty normalized Bahnar text — BLOCKED")

# ==========================================================================
# STEP 1: TRAIN CONTAMINATION AUDIT (on ORIGINAL train)
# ==========================================================================
print("\n" + "="*60)
print("TRAIN CONTAMINATION AUDIT")
print("="*60)

# First build dirty vocab (for audit reference only)
dirty_vocab = build_char_ctc_vocab(train_df["text_bahnar_norm"].tolist())
print(f"Dirty vocab size (before cleaning): {len(dirty_vocab)}")

# Audit train for contamination using vocab-based audit
contam_chars, contam_rows = audit_train_script_contamination(dirty_vocab, train_df=train_df)
write_dataframe_csv(contam_chars, PATHS["audit"] / "notebook02_train_contamination_characters.csv", metadata=RUN_META)
write_dataframe_csv(contam_rows, PATHS["audit"] / "notebook02_train_contamination_rows.csv", metadata=RUN_META)

TRAIN_CONTAMINATION_SUMMARY = compute_contamination_summary(contam_chars, contam_rows)
print(f"Train contamination: {TRAIN_CONTAMINATION_SUMMARY['unexpected_script_characters']} unexpected chars")
print(f"Train contamination: {TRAIN_CONTAMINATION_SUMMARY['affected_rows']} affected rows")
print(f"Train contamination: {TRAIN_CONTAMINATION_SUMMARY['affected_groups']} affected groups")

if TRAIN_CONTAMINATION_SUMMARY["unexpected_script_characters"] > 0:
    print("\nUnexpected script characters in train:")
    display(contam_chars.loc[contam_chars["flags"].str.contains("unexpected_script", na=False)].head(20))

# ==========================================================================
# STEP 2: VALIDATION CONTAMINATION AUDIT (independent, same classifier)
# ==========================================================================
print("\n" + "="*60)
print("VALIDATION CONTAMINATION AUDIT")
print("="*60)

val_contam_chars, val_contam_rows = audit_dataframe_contamination(
    val_df, text_col="text_bahnar", split_name="validation"
)
write_dataframe_csv(val_contam_chars, PATHS["audit"] / "notebook02_validation_contamination_characters.csv", metadata=RUN_META)
write_dataframe_csv(val_contam_rows, PATHS["audit"] / "notebook02_validation_contamination_rows.csv", metadata=RUN_META)

VAL_CONTAMINATION_SUMMARY = compute_contamination_summary(val_contam_chars, val_contam_rows)
print(f"Validation contamination: {VAL_CONTAMINATION_SUMMARY['unexpected_script_characters']} unexpected chars")
print(f"Validation contamination: {VAL_CONTAMINATION_SUMMARY['affected_rows']} affected rows")

if VAL_CONTAMINATION_SUMMARY["unexpected_script_characters"] > 0:
    print("\nUnexpected script characters in validation:")
    display(val_contam_chars.loc[val_contam_chars["flags"].str.contains("unexpected_script", na=False)].head(10))

# ==========================================================================
# STEP 3: BUILD EXCLUSION LISTS AND CLEAN SPLITS
# ==========================================================================
print("\n" + "="*60)
print("BUILDING EXCLUSION LISTS AND CLEAN SPLITS")
print("="*60)

# Build train exclusion list
train_exclusion_df = build_exclusion_list(contam_rows, filter_flags=["unexpected_script"])
write_dataframe_csv(train_exclusion_df, PATHS["audit"] / "notebook02_train_contamination_exclusions.csv", metadata=RUN_META)
print(f"Train exclusions: {len(train_exclusion_df)} records")

# Build validation exclusion list
val_exclusion_df = build_exclusion_list(val_contam_rows, filter_flags=["unexpected_script"])
write_dataframe_csv(val_exclusion_df, PATHS["audit"] / "notebook02_validation_contamination_exclusions.csv", metadata=RUN_META)
print(f"Validation exclusions: {len(val_exclusion_df)} records")

# Build clean splits
train_clean_df = build_clean_split(train_df, train_exclusion_df)
val_clean_df = build_clean_split(val_df, val_exclusion_df)

print(f"\nOriginal train: {len(train_df)}")
print(f"Clean train: {len(train_clean_df)} (excluded {len(train_df) - len(train_clean_df)})")
print(f"Original validation: {len(val_df)}")
print(f"Clean validation: {len(val_clean_df)} (excluded {len(val_df) - len(val_clean_df)})")

# Verify clean splits have no contamination
train_clean_verify = verify_clean_split_no_contamination(train_clean_df, split_name="train_clean")
val_clean_verify = verify_clean_split_no_contamination(val_clean_df, split_name="validation_clean")

print(f"\nTrain clean verification: {'PASS' if train_clean_verify['passed'] else 'FAIL'}")
print(f"Validation clean verification: {'PASS' if val_clean_verify['passed'] else 'FAIL'}")

if not train_clean_verify["passed"]:
    raise RuntimeError(f"Train clean split still has contamination: {train_clean_verify}")
if not val_clean_verify["passed"]:
    raise RuntimeError(f"Validation clean split still has contamination: {val_clean_verify}")

# ==========================================================================
# STEP 3.5: CREATE UID SETS FOR ASSERTIONS
# ==========================================================================
train_excluded_uids = set(train_exclusion_df["record_uid"].astype(str)) if len(train_exclusion_df) > 0 else set()
validation_excluded_uids = set(val_exclusion_df["record_uid"].astype(str)) if len(val_exclusion_df) > 0 else set()
train_clean_uids = set(train_clean_df["record_uid"].astype(str))
validation_clean_uids = set(val_clean_df["record_uid"].astype(str))

print(f"\nUID sets created:")
print(f"  train_excluded_uids: {len(train_excluded_uids)}")
print(f"  validation_excluded_uids: {len(validation_excluded_uids)}")
print(f"  train_clean_uids: {len(train_clean_uids)}")
print(f"  validation_clean_uids: {len(validation_clean_uids)}")

# Verify no overlap between clean and excluded
assert train_clean_uids.isdisjoint(train_excluded_uids), "Train clean/excluded overlap!"
assert validation_clean_uids.isdisjoint(validation_excluded_uids), "Validation clean/excluded overlap!"

# ==========================================================================
# STEP 3.6: COMPUTE ORDERED AND SET HASHES
# ==========================================================================
# Ordered hash preserves DataFrame order, set hash is order-independent
TRAIN_CLEAN_ORDERED_HASH = compute_ordered_uid_hash(train_clean_df)
TRAIN_CLEAN_SET_HASH = compute_uid_set_hash(train_clean_df)
VAL_CLEAN_ORDERED_HASH = compute_ordered_uid_hash(val_clean_df)
VAL_CLEAN_SET_HASH = compute_uid_set_hash(val_clean_df)

# Exclusion file SHAs (for contract)
train_exc_path = PATHS["audit"] / "notebook02_train_contamination_exclusions.csv"
val_exc_path = PATHS["audit"] / "notebook02_validation_contamination_exclusions.csv"
TRAIN_EXCLUSION_CSV_SHA = sha256_file(train_exc_path) if train_exc_path.exists() else hashlib.sha256(b"").hexdigest()
VAL_EXCLUSION_CSV_SHA = sha256_file(val_exc_path) if val_exc_path.exists() else hashlib.sha256(b"").hexdigest()

print(f"\nTrain clean ordered hash: {TRAIN_CLEAN_ORDERED_HASH[:16]}...")
print(f"Train clean set hash: {TRAIN_CLEAN_SET_HASH[:16]}...")
print(f"Validation clean ordered hash: {VAL_CLEAN_ORDERED_HASH[:16]}...")
print(f"Validation clean set hash: {VAL_CLEAN_SET_HASH[:16]}...")

# ==========================================================================
# STEP 4: BUILD VOCABULARY FROM CLEAN TRAIN ONLY
# ==========================================================================
print("\n" + "="*60)
print("BUILDING VOCABULARY FROM CLEAN TRAIN")
print("="*60)

# Normalize clean train texts
train_clean_df["text_bahnar_norm"] = train_clean_df["text_bahnar"].map(normalize_bahnar_ctc_v1)
val_clean_df["text_bahnar_norm"] = val_clean_df["text_bahnar"].map(normalize_bahnar_ctc_v1)

# Build vocab from CLEAN train only
vocab = build_char_ctc_vocab(train_clean_df["text_bahnar_norm"].tolist())
print(f"Clean vocab size: {len(vocab)}")

# Verify vocab has no unexpected scripts
vocab_chars = set(vocab.keys()) - {"[PAD]", "[UNK]", "|"}
unexpected_in_vocab = []
for ch in vocab_chars:
    if classify_character(ch) == "unexpected_script":
        unexpected_in_vocab.append(ch)

if unexpected_in_vocab:
    raise RuntimeError(f"Clean vocab still has unexpected scripts: {unexpected_in_vocab}")
print(f"Vocabulary verification: NO unexpected_script characters")

# ==========================================================================
# STEP 5: SAVE TOKENIZER FROM CLEAN VOCAB
# ==========================================================================
print("\n" + "="*60)
print("SAVING TOKENIZER")
print("="*60)

tok_dir = PATHS["artifacts_tokenizer"]

# Build provenance for tokenizer
tokenizer_provenance = {
    "run_id": RUN_ID,
    "dataset_id": DATASET_ID,
    "dataset_revision": EXPECTED_DATASET_REVISION,
    "normalization_version": NORMALIZATION_VERSION,
    "train_clean_count": len(train_clean_df),
    "train_clean_ordered_uid_sha256": TRAIN_CLEAN_ORDERED_HASH,
    "train_clean_uid_set_sha256": TRAIN_CLEAN_SET_HASH,
    "train_exclusion_csv_sha256": TRAIN_EXCLUSION_CSV_SHA,
}

tok_verification = save_tokenizer_clean(
    vocab, tok_dir, 
    target_sampling_rate=TARGET_SAMPLING_RATE,
    provenance=tokenizer_provenance,
)
print(f"Tokenizer saved with provenance: passed={tok_verification['passed']}")
print(f"  vocab_exact_match: {tok_verification.get('vocab_exact_match')}")
print(f"  vocab_no_unexpected_scripts: {tok_verification.get('vocab_no_unexpected_scripts')}")

# Reload tokenizer to verify
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(str(tok_dir))
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(str(tok_dir))
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

# Verify tokenizer state
assert len(tokenizer) == len(vocab), f"Tokenizer size mismatch: {len(tokenizer)} != {len(vocab)}"
assert tokenizer.pad_token_id == vocab["[PAD]"] == 0
assert tokenizer.bos_token is None, f"BOS should be None, got {tokenizer.bos_token}"
assert tokenizer.eos_token is None, f"EOS should be None, got {tokenizer.eos_token}"
reloaded_vocab = tokenizer.get_vocab()
assert "<s>" not in reloaded_vocab, "<s> found in reloaded vocab"
assert "</s>" not in reloaded_vocab, "</s> found in reloaded vocab"
print(f"Tokenizer verified: size={len(tokenizer)}, pad_id=0, BOS=None, EOS=None, no stale tokens")

# Verify tokenizer provenance
tokenizer_provenance_verification = verify_tokenizer_provenance(
    tokenizer_dir=PATHS["artifacts_tokenizer"],
    expected_run_id=RUN_ID,
    expected_revision=EXPECTED_DATASET_REVISION,
    expected_train_clean_ordered_uid_sha256=TRAIN_CLEAN_ORDERED_HASH,
    expected_train_clean_uid_set_sha256=TRAIN_CLEAN_SET_HASH,
)
print(f"Tokenizer provenance verification: {'PASS' if tokenizer_provenance_verification['passed'] else 'FAIL'}")
if not tokenizer_provenance_verification["passed"]:
    print(f"Errors: {tokenizer_provenance_verification['errors']}")
    raise RuntimeError(f"Tokenizer provenance verification FAILED: {tokenizer_provenance_verification['errors']}")

# Character frequency (from clean train)
freq = character_frequency(train_clean_df["text_bahnar_norm"].tolist())
write_dataframe_csv(freq, PATHS["audit"] / "notebook02_train_character_frequency.csv", metadata=RUN_META)

# ==========================================================================
# STEP 6: VALIDATION OOV WITH CLEAN TOKENIZER AND CLEAN VALIDATION
# ==========================================================================
print("\n" + "="*60)
print("VALIDATION OOV AUDIT (clean tokenizer + clean validation)")
print("="*60)

oov = find_oov_characters(val_clean_df["text_bahnar_norm"].tolist(), vocab)
write_dataframe_csv(oov, PATHS["audit"] / "notebook02_validation_oov_characters.csv", metadata=RUN_META)
oov_rows = find_oov_rows(val_clean_df, vocab, text_col="text_bahnar")
write_dataframe_csv(oov_rows, PATHS["audit"] / "notebook02_validation_oov_rows.csv", metadata=RUN_META)
print(f"Validation OOV characters: {len(oov)}; OOV rows: {len(oov_rows)}")

# ==========================================================================
# EXPORT CLEAN SPLIT CONTRACT
# ==========================================================================
print("\n" + "="*60)
print("EXPORTING CLEAN SPLIT CONTRACT")
print("="*60)

clean_split_contract = export_clean_split_contract(
    output_path=PATHS["results"] / "notebook02_clean_split_contract.json",
    run_id=RUN_ID,
    dataset_id=DATASET_ID,
    dataset_revision=EXPECTED_DATASET_REVISION,
    normalization_version=NORMALIZATION_VERSION,
    base_manifest_sha256=EXPECTED_MANIFEST_SHA256,
    train_original_count=len(train_df),
    train_exclusion_count=len(train_exclusion_df),
    train_clean_count=len(train_clean_df),
    train_ordered_uid_sha256=TRAIN_CLEAN_ORDERED_HASH,
    train_uid_set_sha256=TRAIN_CLEAN_SET_HASH,
    train_exclusion_csv_sha256=TRAIN_EXCLUSION_CSV_SHA,
    validation_original_count=len(val_df),
    validation_exclusion_count=len(val_exclusion_df),
    validation_clean_count=len(val_clean_df),
    validation_ordered_uid_sha256=VAL_CLEAN_ORDERED_HASH,
    validation_uid_set_sha256=VAL_CLEAN_SET_HASH,
    validation_exclusion_csv_sha256=VAL_EXCLUSION_CSV_SHA,
)
print(f"Clean split contract exported: {PATHS['results'] / 'notebook02_clean_split_contract.json'}")

# Verify clean split contract immediately after export
clean_split_contract_verification = verify_clean_split_contract(
    contract_path=PATHS["results"] / "notebook02_clean_split_contract.json",
    expected_run_id=RUN_ID,
    expected_revision=EXPECTED_DATASET_REVISION,
    train_clean_df=train_clean_df,
    validation_clean_df=val_clean_df,
    train_exclusion_path=PATHS["audit"] / "notebook02_train_contamination_exclusions.csv",
    validation_exclusion_path=PATHS["audit"] / "notebook02_validation_contamination_exclusions.csv",
)
print(f"Clean split contract verification: {'PASS' if clean_split_contract_verification['passed'] else 'FAIL'}")
if not clean_split_contract_verification["passed"]:
    print(f"Errors: {clean_split_contract_verification['errors']}")
    raise RuntimeError(f"Clean split contract verification FAILED: {clean_split_contract_verification['errors']}")

# ==========================================================================
# CONTAMINATION SUMMARY (combined) - NO manual approval needed
# ==========================================================================
# Contamination is automatically remediated by exclusion lists
contamination_remediated = (
    train_clean_verify["passed"] and 
    val_clean_verify["passed"] and
    len(unexpected_in_vocab) == 0
)

CONTAMINATION_SUMMARY = {
    "train": TRAIN_CONTAMINATION_SUMMARY,
    "validation": VAL_CONTAMINATION_SUMMARY,
    "unexpected_script_characters": TRAIN_CONTAMINATION_SUMMARY["unexpected_script_characters"],
    "affected_rows": TRAIN_CONTAMINATION_SUMMARY["affected_rows"],
    "affected_groups": TRAIN_CONTAMINATION_SUMMARY["affected_groups"],
    "train_exclusions": len(train_exclusion_df),
    "validation_exclusions": len(val_exclusion_df),
    "train_clean_rows": len(train_clean_df),
    "validation_clean_rows": len(val_clean_df),
    "clean_vocab_size": len(vocab),
    "remediated": contamination_remediated,  # Auto-approved via exclusion
}

# ==========================================================================
# TEXT CHECKS
# ==========================================================================
text_checks = pd.DataFrame([
    {"check": "vocab_from_clean_train_only", "passed": True},
    {"check": "vocab_no_unexpected_scripts", "passed": len(unexpected_in_vocab) == 0},
    {"check": "pad_id_zero", "passed": vocab.get("[PAD]") == 0},
    {"check": "has_unk_and_word_delim", "passed": "[UNK]" in vocab and "|" in vocab},
    {"check": "no_space_token", "passed": " " not in vocab},
    {"check": "train_norm_non_empty", "passed": empty_train == 0},
    {"check": "tokenizer_size_matches_vocab", "passed": len(tokenizer) == len(vocab)},
    {"check": "tokenizer_no_bos_eos", "passed": tokenizer.bos_token is None and tokenizer.eos_token is None},
    {"check": "tokenizer_no_stale_tokens", "passed": "<s>" not in reloaded_vocab and "</s>" not in reloaded_vocab},
    {"check": "ctc_blank_is_pad_zero", "passed": tokenizer.pad_token_id == 0},
    {"check": "train_contamination_audit_done", "passed": True},
    {"check": "validation_contamination_audit_done", "passed": True},
    {"check": "train_clean_no_contamination", "passed": train_clean_verify["passed"]},
    {"check": "validation_clean_no_contamination", "passed": val_clean_verify["passed"]},
    {"check": "exclusion_lists_exported", "passed": True},
    {"check": "validation_oov_exported", "passed": True},
    {"check": "frozen_test_not_used_for_vocab", "passed": True},
])
write_dataframe_csv(text_checks, PATHS["audit"] / "notebook02_text_checks.csv", metadata=RUN_META)
display(text_checks)
if not text_checks["passed"].all():
    failed = text_checks.loc[~text_checks["passed"], "check"].tolist()
    raise RuntimeError(f"Text/vocab checks FAILED: {failed}")


Empty normalized train texts: 0
Empty normalized validation texts: 0

TRAIN CONTAMINATION AUDIT
Dirty vocab size (before cleaning): 210
Train contamination: 70 unexpected chars
Train contamination: 191 affected rows
Train contamination: 129 affected groups

Unexpected script characters in train:


,character,codepoint,unicode_name,category,vocab_id,classification,flags,occurrence_count,affected_row_count
0,ә,U+04D9,CYRILLIC SMALL LETTER SCHWA,Ll,89,unexpected_script,unexpected_script,4,1
1,צ,U+05E6,HEBREW LETTER TSADI,Lo,90,unexpected_script,unexpected_script,1,1
2,ง,U+0E07,THAI CHARACTER NGO NGU,Lo,91,unexpected_script,unexpected_script,1,1
3,ต,U+0E15,THAI CHARACTER TO TAO,Lo,92,unexpected_script,unexpected_script,1,1
4,ว,U+0E27,THAI CHARACTER WO WAEN,Lo,93,unexpected_script,unexpected_script,1,1
5,ั,U+0E31,THAI CHARACTER MAI HAN-AKAT,Mn,94,unexpected_script,unexpected_script,1,1
6,ក,U+1780,KHMER LETTER KA,Lo,95,unexpected_script,unexpected_script,1163,105
7,ខ,U+1781,KHMER LETTER KHA,Lo,96,unexpected_script,unexpected_script,110,55
8,គ,U+1782,KHMER LETTER KO,Lo,97,unexpected_script,unexpected_script,215,83
9,ឃ,U+1783,KHMER LETTER KHO,Lo,98,unexpected_script,unexpected_script,25,16



VALIDATION CONTAMINATION AUDIT
Validation contamination: 56 unexpected chars
Validation contamination: 19 affected rows

Unexpected script characters in validation:


,character,codepoint,unicode_name,category,classification,flags,occurrence_count,affected_row_count
0,្,U+17D2,KHMER SIGN COENG,Mn,unexpected_script,unexpected_script,174,12
1,ា,U+17B6,KHMER VOWEL SIGN AA,Mc,unexpected_script,unexpected_script,171,12
2,រ,U+179A,KHMER LETTER RO,Lo,unexpected_script,unexpected_script,140,12
3,ក,U+1780,KHMER LETTER KA,Lo,unexpected_script,unexpected_script,109,12
4,ង,U+1784,KHMER LETTER NGO,Lo,unexpected_script,unexpected_script,108,14
5,ត,U+178F,KHMER LETTER TA,Lo,unexpected_script,unexpected_script,102,12
6,ន,U+1793,KHMER LETTER NO,Lo,unexpected_script,unexpected_script,93,12
7,ប,U+1794,KHMER LETTER BA,Lo,unexpected_script,unexpected_script,89,12
8,ម,U+1798,KHMER LETTER MO,Lo,unexpected_script,unexpected_script,74,12
9,ច,U+1785,KHMER LETTER CA,Lo,unexpected_script,unexpected_script,71,13



BUILDING EXCLUSION LISTS AND CLEAN SPLITS
Train exclusions: 191 records
Validation exclusions: 19 records

Original train: 102698
Clean train: 102507 (excluded 191)
Original validation: 11132
Clean validation: 11113 (excluded 19)

Train clean verification: PASS
Validation clean verification: PASS

UID sets created:
  train_excluded_uids: 191
  validation_excluded_uids: 19
  train_clean_uids: 102507
  validation_clean_uids: 11113

Train clean ordered hash: 1fae58ebf435557f...
Train clean set hash: 5672eaef7e02670c...
Validation clean ordered hash: 532b7f51dd66508c...
Validation clean set hash: 6b9a55458734ff7c...

BUILDING VOCABULARY FROM CLEAN TRAIN
Clean vocab size: 140
Vocabulary verification: NO unexpected_script characters

SAVING TOKENIZER
Tokenizer saved with provenance: passed=True
  vocab_exact_match: True
  vocab_no_unexpected_scripts: True
Tokenizer verified: size=140, pad_id=0, BOS=None, EOS=None, no stale tokens
Tokenizer provenance verification: PASS

VALIDATION OOV AUDIT

,check,passed
0,vocab_from_clean_train_only,True
1,vocab_no_unexpected_scripts,True
2,pad_id_zero,True
3,has_unk_and_word_delim,True
4,no_space_token,True
5,train_norm_non_empty,True
6,tokenizer_size_matches_vocab,True
7,tokenizer_no_bos_eos,True
8,tokenizer_no_stale_tokens,True
9,ctc_blank_is_pad_zero,True


## Step 3 — Sampled waveform QA qua Dataset Viewer `/rows`

**Không** stream 113k rows. Dùng offset phân bố + window nhỏ.

500 train + 70 validation chỉ là **sample QA** (duration 0.5–30 s, buffer cho Notebook 03), không phải toàn bộ train audio.


In [6]:
# Cell 6 — Viewer windows → candidate pool (with planned counts tracking)
size_payload = dataset_viewer_get(
    "size",
    params={"dataset": DATASET_ID, "config": CONFIG_NAME},
    token=HF_TOKEN,
    session=VIEWER_SESSION,
    pace_seconds=0.1,
)
orig_train_rows = None
for item in size_payload.get("size", {}).get("splits", []):
    if item.get("config") == CONFIG_NAME and item.get("split") == "train":
        orig_train_rows = int(item["num_rows"])
        break
if not orig_train_rows:
    raise RuntimeError("Could not read original train size from Dataset Viewer /size")
print(f"Original HF train rows: {orig_train_rows:,}")

offsets = uniform_window_offsets(orig_train_rows, window_length=VIEWER_WINDOW_LENGTH, max_windows=VIEWER_MAX_WINDOWS, seed=SEED)
assert offsets[0] == 0 and offsets[-1] == max(0, orig_train_rows - VIEWER_WINDOW_LENGTH)
print(f"Viewer windows: {len(offsets)} x length={VIEWER_WINDOW_LENGTH}")

# CRITICAL: Use CLEAN splits for candidate selection (excludes contaminated records)
train_key = {("train", str(r)): i for i, r in enumerate(train_clean_df["record_id"].astype(str))}
val_key = {("train", str(r)): i for i, r in enumerate(val_clean_df["record_id"].astype(str))}

candidate_rows = []
for off in offsets:
    payload = dataset_viewer_get(
        "rows",
        params={"dataset": DATASET_ID, "config": CONFIG_NAME, "split": "train", "offset": int(off), "length": int(VIEWER_WINDOW_LENGTH)},
        token=HF_TOKEN,
        session=VIEWER_SESSION,
        pace_seconds=1.5,
    )
    for j, row in enumerate(payload.get("rows", [])):
        row_idx = int(row.get("row_idx")) if isinstance(row, dict) and row.get("row_idx") is not None else int(off) + j
        cell = row.get("row") if isinstance(row, dict) else row
        if not isinstance(cell, dict):
            continue
        rid = cell.get("id")
        if rid is None:
            continue
        rid = str(rid)
        key = ("train", rid)
        if key in train_key:
            final_split = "train"
            m = train_clean_df.iloc[train_key[key]]
        elif key in val_key:
            final_split = "validation"
            m = val_clean_df.iloc[val_key[key]]
        else:
            continue  # Not in clean splits (excluded or not found)
        audio_src = extract_audio_src(cell.get("audio"))
        remote_text = cell.get("text_bahnar") or cell.get("transcript") or cell.get("sentence_bahnar")
        identity_ok = texts_match_after_light_norm(m["text_bahnar"], remote_text)
        candidate_rows.append({
            "final_split": final_split, "record_uid": m["record_uid"], "record_id": rid,
            "source_label": m["source_label"], "group_id": m["group_id"],
            "recording_group_id": m["recording_group_id"], "duration_seconds": m["duration_seconds"],
            "duration_bin": duration_bin(m["duration_seconds"]),
            "manifest_text_bahnar": m["text_bahnar"], "remote_text_bahnar": remote_text,
            "text_identity_ok": identity_ok, "row_idx": row_idx, "viewer_offset": int(off),
            "_audio_src": audio_src,
        })

candidates = pd.DataFrame(candidate_rows)
cand_train_raw = candidates.loc[candidates["final_split"] == "train"].copy()
cand_val_raw = candidates.loc[candidates["final_split"] == "validation"].copy()

# BEFORE sampling/cache: keep only Notebook-03-compatible durations (0.5–30 s).
cand_train = filter_candidates_by_duration(
    cand_train_raw,
    min_duration=CACHE_SAMPLE_MIN_DURATION,
    max_duration=CACHE_SAMPLE_MAX_DURATION,
)
cand_val = filter_candidates_by_duration(
    cand_val_raw,
    min_duration=CACHE_SAMPLE_MIN_DURATION,
    max_duration=CACHE_SAMPLE_MAX_DURATION,
)
unique_train = cand_train["record_uid"].nunique()
unique_val = cand_val["record_uid"].nunique()

print(
    f"Candidate pool (raw): {len(candidates)} "
    f"(train unique={cand_train_raw['record_uid'].nunique()}, "
    f"val unique={cand_val_raw['record_uid'].nunique()})"
)
print(
    f"Candidate pool (duration {CACHE_SAMPLE_MIN_DURATION}–{CACHE_SAMPLE_MAX_DURATION}s): "
    f"train unique={unique_train} (dropped {len(cand_train_raw) - len(cand_train)}), "
    f"val unique={unique_val} (dropped {len(cand_val_raw) - len(cand_val)})"
)
if unique_train < QA_TRAIN_TARGET:
    raise RuntimeError(
        f"Insufficient train candidates after duration filter: "
        f"unique={unique_train} < target={QA_TRAIN_TARGET}. "
        f"Increase VIEWER_MAX_WINDOWS or broaden windows."
    )
if unique_val < QA_VALIDATION_TARGET:
    raise RuntimeError(
        f"Insufficient validation candidates after duration filter: "
        f"unique={unique_val} < target={QA_VALIDATION_TARGET}. "
        f"Increase VIEWER_MAX_WINDOWS or broaden windows."
    )

# ==========================================================================
# ASSERTION: ALL CANDIDATES ARE FROM CLEAN SPLITS
# ==========================================================================
candidate_train_uids = set(cand_train["record_uid"].astype(str))
candidate_val_uids = set(cand_val["record_uid"].astype(str))

# Check all candidates are in clean splits
assert candidate_train_uids.issubset(train_clean_uids), \
    f"Train candidates not from clean split: {candidate_train_uids - train_clean_uids}"
assert candidate_val_uids.issubset(validation_clean_uids), \
    f"Validation candidates not from clean split: {candidate_val_uids - validation_clean_uids}"

# Check no candidates are in excluded sets
assert candidate_train_uids.isdisjoint(train_excluded_uids), \
    f"Train candidates contain excluded UIDs: {candidate_train_uids & train_excluded_uids}"
assert candidate_val_uids.isdisjoint(validation_excluded_uids), \
    f"Validation candidates contain excluded UIDs: {candidate_val_uids & validation_excluded_uids}"

print(f"✓ All candidates verified from clean splits (no excluded records)")

# Text identity check (must be 100% on sampled pool)
mismatches = candidates.loc[~candidates["text_identity_ok"].fillna(False)]
SAMPLED_TEXT_IDENTITY_OK = len(mismatches) == 0
SAMPLED_TEXT_IDENTITY_RATE = float(candidates["text_identity_ok"].fillna(False).mean()) if len(candidates) else 0.0
if not SAMPLED_TEXT_IDENTITY_OK:
    mismatch_report = mismatches[["record_uid", "record_id", "final_split", "manifest_text_bahnar", "remote_text_bahnar"]].copy()
    mismatch_report.columns = ["record_uid", "record_id", "source_split", "expected_text", "observed_text"]
    write_dataframe_csv(mismatch_report, PATHS["audit"] / "notebook02_text_identity_mismatches.csv", metadata=RUN_META)
    print(f"TEXT IDENTITY MISMATCHES: {len(mismatch_report)} → exported")
else:
    print("Sampled text identity: 100%")

# Planned counts = min(target, unique available)
PLANNED_QA_TRAIN = min(QA_TRAIN_TARGET, unique_train)
PLANNED_QA_VALIDATION = min(QA_VALIDATION_TARGET, unique_val)

print(f"PLANNED counts: train={PLANNED_QA_TRAIN} (target={QA_TRAIN_TARGET}, avail={unique_train})")
print(f"PLANNED counts: val={PLANNED_QA_VALIDATION} (target={QA_VALIDATION_TARGET}, avail={unique_val})")

# Select representative samples
qa_train_sel = select_representative_samples(cand_train, n=PLANNED_QA_TRAIN, seed=SEED)
qa_val_sel = select_representative_samples(cand_val, n=PLANNED_QA_VALIDATION, seed=SEED + 1)

# CRITICAL: Assert selected == planned
assert len(qa_train_sel) == PLANNED_QA_TRAIN, f"Train selection mismatch: {len(qa_train_sel)} != {PLANNED_QA_TRAIN}"
assert len(qa_val_sel) == PLANNED_QA_VALIDATION, f"Val selection mismatch: {len(qa_val_sel)} != {PLANNED_QA_VALIDATION}"
assert len(qa_train_sel) == QA_TRAIN_TARGET, f"Train target not met: {len(qa_train_sel)} != {QA_TRAIN_TARGET}"
assert len(qa_val_sel) == QA_VALIDATION_TARGET, f"Val target not met: {len(qa_val_sel)} != {QA_VALIDATION_TARGET}"
# All selected samples must be within the Notebook 03 duration gate.
_train_durs = pd.to_numeric(qa_train_sel["duration_seconds"], errors="coerce")
_val_durs = pd.to_numeric(qa_val_sel["duration_seconds"], errors="coerce")
assert ((_train_durs >= CACHE_SAMPLE_MIN_DURATION) & (_train_durs <= CACHE_SAMPLE_MAX_DURATION)).all()
assert ((_val_durs >= CACHE_SAMPLE_MIN_DURATION) & (_val_durs <= CACHE_SAMPLE_MAX_DURATION)).all()
print(f"Selected: train={len(qa_train_sel)} val={len(qa_val_sel)} (matches planned; duration-filtered)")

qa_selected = pd.concat([qa_train_sel, qa_val_sel], ignore_index=True)
assert qa_selected["record_uid"].nunique() == len(qa_selected)

# ==========================================================================
# ASSERTION: ALL QA SAMPLES ARE FROM CLEAN SPLITS
# ==========================================================================
qa_train_uids = set(qa_train_sel["record_uid"].astype(str))
qa_val_uids = set(qa_val_sel["record_uid"].astype(str))

assert qa_train_uids.issubset(train_clean_uids), \
    f"QA train samples not from clean split: {qa_train_uids - train_clean_uids}"
assert qa_val_uids.issubset(validation_clean_uids), \
    f"QA validation samples not from clean split: {qa_val_uids - validation_clean_uids}"
assert qa_train_uids.isdisjoint(train_excluded_uids), \
    f"QA train contains excluded: {qa_train_uids & train_excluded_uids}"
assert qa_val_uids.isdisjoint(validation_excluded_uids), \
    f"QA validation contains excluded: {qa_val_uids & validation_excluded_uids}"

print(f"✓ All QA samples verified from clean splits")

# Save candidates (no signed URLs)
to_save = strip_signed_url_columns(qa_selected.copy())
write_dataframe_csv(to_save, PATHS["audit"] / "notebook02_audio_candidates.csv", metadata=RUN_META)

# Distribution report
dist = sample_distribution_report(qa_selected)
write_dataframe_csv(dist, PATHS["audit"] / "notebook02_audio_sample_distribution.csv", metadata=RUN_META)
display(dist.head(30))


Original HF train rows: 113,830
Viewer windows: 25 x length=100
Candidate pool (raw): 2495 (train unique=2236, val unique=259)
Candidate pool (duration 0.5–30.0s): train unique=1823 (dropped 413), val unique=212 (dropped 47)
✓ All candidates verified from clean splits (no excluded records)
Sampled text identity: 100%
PLANNED counts: train=500 (target=500, avail=1823)
PLANNED counts: val=70 (target=70, avail=212)
Selected: train=500 val=70 (matches planned; duration-filtered)
✓ All QA samples verified from clean splits


,source_label,duration_bin,n_rows,n_groups
0,KT_0,15-30s,14,8
1,KT_0,3-8s,15,8
2,KT_0,8-15s,16,8
3,yt_-1,0-3s,5,5
4,yt_-1,15-30s,10,8
5,yt_-1,3-8s,14,12
6,yt_-1,8-15s,10,8
7,yt_24,15-30s,6,2
8,yt_24,3-8s,2,1
9,yt_24,8-15s,2,2


## Step 4 — Download / decode / QA / cache 16 kHz mono

Reuse `check_waveform`. Cache WAV PCM16. Rerun-safe. Không lưu signed URL.


In [7]:
# Cell 7 — Source QA vs cache QA using process_audio_candidate helper
from __future__ import annotations
import io
import soundfile as sf

def refresh_audio_src(row_idx: int) -> str | None:
    """Refresh signed URL from Dataset Viewer."""
    off = (int(row_idx) // VIEWER_WINDOW_LENGTH) * VIEWER_WINDOW_LENGTH
    payload = dataset_viewer_get(
        "rows",
        params={"dataset": DATASET_ID, "config": CONFIG_NAME, "split": "train", "offset": off, "length": VIEWER_WINDOW_LENGTH},
        token=HF_TOKEN, session=VIEWER_SESSION, pace_seconds=0.5,
    )
    rows = payload.get("rows", [])
    local = int(row_idx) - off
    if 0 <= local < len(rows):
        cell = rows[local].get("row") if isinstance(rows[local], dict) else rows[local]
        return extract_audio_src(cell.get("audio") if isinstance(cell, dict) else None)
    return None

# Process all selected samples using the tested helper
qa_records = []
fresh_count = 0
reuse_count = 0

for i, (_, row) in enumerate(qa_selected.iterrows()):
    if (i + 1) % 25 == 0 or i == 0:
        print(f"  audio QA progress {i+1}/{len(qa_selected)}")
    
    result = process_audio_candidate(
        record_uid=str(row["record_uid"]),
        record_id=row["record_id"],
        final_split=row["final_split"],
        source_label=row["source_label"],
        group_id=row["group_id"],
        duration_seconds_meta=row["duration_seconds"],
        text_identity_ok=bool(row["text_identity_ok"]),
        audio_src=row.get("_audio_src"),
        row_idx=int(row["row_idx"]),
        cache_dir=PATHS["cache_audio"],
        project_root=PROJECT_ROOT,
        dataset_revision=EXPECTED_DATASET_REVISION,
        target_sr=TARGET_SAMPLING_RATE,
        min_duration=AUDIO_MIN_DURATION,
        max_duration=AUDIO_MAX_DURATION,
        download_fn=lambda url: download_audio_bytes(url, session=ASSET_SESSION),
        refresh_url_fn=refresh_audio_src,
        decode_fn=lambda b: sf.read(io.BytesIO(b), always_2d=False),
    )
    
    if result.get("reused_cache"):
        reuse_count += 1
    else:
        fresh_count += 1
    
    qa_records.append(result)

qa_df = pd.DataFrame(qa_records)
assert_no_signed_urls_persisted(qa_df)

qa_train = qa_df.loc[qa_df["final_split"] == "train"].copy()
qa_val = qa_df.loc[qa_df["final_split"] == "validation"].copy()

# CRITICAL: Assert QA count == planned count
assert len(qa_train) == PLANNED_QA_TRAIN, f"QA train count mismatch: {len(qa_train)} != {PLANNED_QA_TRAIN}"
assert len(qa_val) == PLANNED_QA_VALIDATION, f"QA val count mismatch: {len(qa_val)} != {PLANNED_QA_VALIDATION}"
print(f"QA counts match planned: train={len(qa_train)} val={len(qa_val)}")

# ==========================================================================
# FIX: Compute cache stats INDEPENDENTLY per split from reused_cache column
# ==========================================================================
from src.data_utils import compute_cache_stats_per_split, verify_cache_stats_consistency

CACHE_STATS = compute_cache_stats_per_split(qa_df, split_col="final_split", reused_col="reused_cache")

# Extract per-split stats
train_fresh = CACHE_STATS.get("train", {}).get("fresh_count", 0)
train_reuse = CACHE_STATS.get("train", {}).get("reuse_count", 0)
val_fresh = CACHE_STATS.get("validation", {}).get("fresh_count", 0)
val_reuse = CACHE_STATS.get("validation", {}).get("reuse_count", 0)

# Verify consistency
cache_verify = verify_cache_stats_consistency(
    CACHE_STATS,
    expected_totals={"train": len(qa_train), "validation": len(qa_val)}
)
if not cache_verify["passed"]:
    raise RuntimeError(f"Cache stats consistency check FAILED: {cache_verify['errors']}")

# Global stats (should equal sum of split stats)
global_fresh = cache_verify["global_fresh"]
global_reuse = cache_verify["global_reuse"]

print(f"Cache stats per split:")
print(f"  train: n={len(qa_train)}, fresh={train_fresh}, reused={train_reuse}")
print(f"  validation: n={len(qa_val)}, fresh={val_fresh}, reused={val_reuse}")
print(f"  global: fresh={global_fresh}, reused={global_reuse}")

# Assertions to catch future regressions
assert train_fresh + train_reuse == len(qa_train), "Train cache stats don't sum to total"
assert val_fresh + val_reuse == len(qa_val), "Validation cache stats don't sum to total"
assert global_fresh == train_fresh + val_fresh, "Global fresh != sum of split fresh"
assert global_reuse == train_reuse + val_reuse, "Global reuse != sum of split reuse"

write_dataframe_csv(qa_train, PATHS["audit"] / "notebook02_train_audio_qa.csv", metadata=RUN_META)
write_dataframe_csv(qa_val, PATHS["audit"] / "notebook02_validation_audio_qa.csv", metadata=RUN_META)

def _rate(df, col="qa_hard_ok"):
    return float(df[col].fillna(False).map(parse_bool).mean()) if len(df) else 0.0

# Build audio summary with CORRECT per-split cache stats
audio_summary = pd.DataFrame([
    {"split": "train", "n": len(qa_train), "hard_ok_rate": _rate(qa_train),
     "hard_ok_n": int(qa_train["qa_hard_ok"].fillna(False).map(parse_bool).sum()),
     "source_hard_ok_rate": _rate(qa_train, "source_hard_ok"),
     "cache_hard_ok_rate": _rate(qa_train, "cache_hard_ok"),
     "fresh_count": train_fresh, "reuse_count": train_reuse},  # FIXED: per-split stats
    {"split": "validation", "n": len(qa_val), "hard_ok_rate": _rate(qa_val),
     "hard_ok_n": int(qa_val["qa_hard_ok"].fillna(False).map(parse_bool).sum()),
     "source_hard_ok_rate": _rate(qa_val, "source_hard_ok"),
     "cache_hard_ok_rate": _rate(qa_val, "cache_hard_ok"),
     "fresh_count": val_fresh, "reuse_count": val_reuse},  # FIXED: per-split stats
])
write_dataframe_csv(audio_summary, PATHS["audit"] / "notebook02_audio_qa_summary.csv", metadata=RUN_META)
display(audio_summary)

HUB_SHA_AFTER_QA = fetch_hub_dataset_sha(DATASET_ID, token=HF_TOKEN)
print(f"Hub SHA (after QA): {HUB_SHA_AFTER_QA}")
if HUB_SHA_AFTER_QA != HUB_SHA_BEFORE or HUB_SHA_AFTER_QA != EXPECTED_DATASET_REVISION:
    raise RuntimeError("Dataset revision changed during QA — BLOCKED")

# QA gates
if qa_df["record_uid"].duplicated().any():
    raise RuntimeError("Duplicate record_uid in QA — FAILED")
train_rate, val_rate = _rate(qa_train), _rate(qa_val)
if train_rate < HARD_OK_RATE_MIN or val_rate < HARD_OK_RATE_MIN:
    raise RuntimeError(f"hard_ok rate below {HARD_OK_RATE_MIN}: train={train_rate:.3f} val={val_rate:.3f} — FAILED")
if not SAMPLED_TEXT_IDENTITY_OK:
    raise RuntimeError(f"Sampled text identity must be 100% (got {SAMPLED_TEXT_IDENTITY_RATE:.6f}) — FAILED")
print("Audio QA gates passed.")


  audio QA progress 1/570
  audio QA progress 25/570
  audio QA progress 50/570
  audio QA progress 75/570
  audio QA progress 100/570
  audio QA progress 125/570
  audio QA progress 150/570
  audio QA progress 175/570
  audio QA progress 200/570
  audio QA progress 225/570
  audio QA progress 250/570
  audio QA progress 275/570
  audio QA progress 300/570
  audio QA progress 325/570
  audio QA progress 350/570
  audio QA progress 375/570
  audio QA progress 400/570
  audio QA progress 425/570
  audio QA progress 450/570
  audio QA progress 475/570
  audio QA progress 500/570
  audio QA progress 525/570
  audio QA progress 550/570
QA counts match planned: train=500 val=70
Cache stats per split:
  train: n=500, fresh=0, reused=500
  validation: n=70, fresh=0, reused=70
  global: fresh=0, reused=570


,split,n,hard_ok_rate,hard_ok_n,source_hard_ok_rate,cache_hard_ok_rate,fresh_count,reuse_count
0,train,500,1.0,500,1.0,1.0,0,500
1,validation,70,1.0,70,1.0,1.0,0,70


Hub SHA (after QA): 3d88d3951b1a6e3388559b341cd7bd274879d696
Audio QA gates passed.


## Step 5 — ASR engineering smoke test (NOT a research result)

Model mặc định: `hf-internal-testing/tiny-random-wav2vec2` — chỉ kiểm tra pipeline.

CER (nếu có) **không** được đưa vào bảng RQ1.


In [8]:
# Cell 8 — Tiny CTC smoke train/infer with PINNED model revision
from __future__ import annotations
from transformers import AutoModelForCTC

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
elif DEVICE == "mps":
    try:
        torch.mps.manual_seed(SEED)
    except Exception:
        pass

ok_train = qa_train.loc[qa_train["qa_hard_ok"].fillna(False).map(parse_bool)].copy()
ok_val = qa_val.loc[qa_val["qa_hard_ok"].fillna(False).map(parse_bool)].copy()
ok_train = ok_train.sort_values("source_duration_sec").head(max(SMOKE_TRAIN_EXAMPLES * 3, SMOKE_TRAIN_EXAMPLES))
ok_val = ok_val.sort_values("source_duration_sec").head(max(SMOKE_VALIDATION_EXAMPLES * 3, SMOKE_VALIDATION_EXAMPLES))

smoke_train = ok_train.head(SMOKE_TRAIN_EXAMPLES)
smoke_val = ok_val.head(SMOKE_VALIDATION_EXAMPLES)
if len(smoke_train) < SMOKE_TRAIN_EXAMPLES or len(smoke_val) < SMOKE_VALIDATION_EXAMPLES:
    raise RuntimeError("Not enough hard_ok cached clips for smoke test — FAILED")

# Ensure no frozen test used
assert "test" not in set(qa_df["final_split"].unique())

# CRITICAL: Use CLEAN splits for text normalization (smoke samples must be from clean)
train_norm_map = train_clean_df.set_index("record_uid")["text_bahnar_norm"].to_dict()
val_norm_map = val_clean_df.set_index("record_uid")["text_bahnar_norm"].to_dict()

# Assert smoke samples are from clean splits
smoke_train_uids = set(smoke_train["record_uid"].astype(str))
smoke_val_uids = set(smoke_val["record_uid"].astype(str))
assert smoke_train_uids.issubset(train_clean_uids), \
    f"Smoke train samples not from clean split: {smoke_train_uids - train_clean_uids}"
assert smoke_val_uids.issubset(validation_clean_uids), \
    f"Smoke val samples not from clean split: {smoke_val_uids - validation_clean_uids}"
print(f"✓ Smoke samples verified from clean splits")

def load_smoke_batch(frame, norm_map):
    waves, texts, uids = [], [], []
    for _, r in frame.iterrows():
        path = PROJECT_ROOT / r["cache_path"]
        wav, sr = read_wav(path)
        assert sr == TARGET_SAMPLING_RATE
        waves.append(to_mono_float32(wav))
        texts.append(norm_map[r["record_uid"]])
        uids.append(r["record_uid"])
    return waves, texts, uids

train_waves, train_texts, train_uids = load_smoke_batch(smoke_train, train_norm_map)
val_waves, val_texts, val_uids = load_smoke_batch(smoke_val, val_norm_map)

def collate(waves, texts):
    audio_inputs = feature_extractor(waves, sampling_rate=TARGET_SAMPLING_RATE, return_tensors="pt", padding=True)
    labels_batch = tokenizer(texts, return_tensors="pt", padding=True)
    labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
    return audio_inputs, labels

# Load model with PINNED REVISION
print(f"Loading smoke model: {SMOKE_MODEL_ID} @ revision={SMOKE_MODEL_REVISION}")
try:
    model = AutoModelForCTC.from_pretrained(
        SMOKE_MODEL_ID, 
        revision=SMOKE_MODEL_REVISION,  # PINNED REVISION
        ctc_loss_reduction="mean", 
        ctc_zero_infinity=True,
        pad_token_id=tokenizer.pad_token_id, 
        vocab_size=len(tokenizer), 
        ignore_mismatched_sizes=True,
    )
except Exception as exc:
    raise RuntimeError(f"Failed to load smoke model {SMOKE_MODEL_ID!r} @ {SMOKE_MODEL_REVISION}: {exc}") from exc

# Verify loaded revision
smoke_revision_requested = SMOKE_MODEL_REVISION
smoke_revision_loaded = getattr(model.config, "_commit_hash", None)
print(f"Requested revision: {smoke_revision_requested}")
print(f"Loaded revision: {smoke_revision_loaded}")

model.config.pad_token_id = tokenizer.pad_token_id
model.to(DEVICE)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

audio_inputs, labels = collate(train_waves, train_texts)
audio_inputs = {k: v.to(DEVICE) for k, v in audio_inputs.items()}
labels = labels.to(DEVICE)

# Forward pass
out = model(input_values=audio_inputs["input_values"], attention_mask=audio_inputs.get("attention_mask"), labels=labels)
loss0 = float(out.loss.detach().cpu())
logits = out.logits
print(f"logits shape: {tuple(logits.shape)}, vocab: {len(tokenizer)}")

assert logits.size(-1) == len(tokenizer), f"Logits vocab dim mismatch"
assert torch.isfinite(logits).all(), "Non-finite logits"
assert np.isfinite(loss0), f"Non-finite loss: {loss0}"

# Training steps
losses = [loss0]
finite_grads = []
for step in range(SMOKE_TRAIN_STEPS):
    optimizer.zero_grad(set_to_none=True)
    out = model(input_values=audio_inputs["input_values"], attention_mask=audio_inputs.get("attention_mask"), labels=labels)
    loss = out.loss
    assert torch.isfinite(loss), "Non-finite loss during training"
    assert torch.isfinite(out.logits).all(), "Non-finite logits during training"
    loss.backward()
    grad_ok = all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    finite_grads.append(grad_ok)
    if not grad_ok:
        raise RuntimeError("Non-finite gradients")
    optimizer.step()
    losses.append(float(loss.detach().cpu()))

assert len(finite_grads) == SMOKE_TRAIN_STEPS == 2, f"Expected 2 optimizer steps, got {len(finite_grads)}"

# Inference
model.eval()
with torch.no_grad():
    val_inputs, _ = collate(val_waves, val_texts)
    val_inputs = {k: v.to(DEVICE) for k, v in val_inputs.items()}
    logits_v = model(input_values=val_inputs["input_values"], attention_mask=val_inputs.get("attention_mask")).logits
    assert torch.isfinite(logits_v).all(), "Non-finite validation logits"
    pred_ids = torch.argmax(logits_v, dim=-1).cpu().numpy()
    predictions = tokenizer.batch_decode(pred_ids)

try:
    from jiwer import cer as jiwer_cer
    tech_cer = float(jiwer_cer(val_texts, predictions))
except Exception:
    tech_cer = None

smoke_report = {
    **RUN_META,
    "device": DEVICE,
    "smoke_model_id": SMOKE_MODEL_ID,
    "smoke_model_revision_requested": smoke_revision_requested,
    "smoke_model_revision_loaded": smoke_revision_loaded,
    "vocabulary_size": len(tokenizer),
    "smoke_train_record_uids": train_uids,
    "smoke_validation_record_uids": val_uids,
    "input_values_shape": list(audio_inputs["input_values"].shape),
    "logits_shape": list(logits.shape),
    "losses": losses,
    "optimizer_steps": SMOKE_TRAIN_STEPS,
    "loss_finite": all(np.isfinite(x) for x in losses),
    "logits_finite": True,
    "gradients_finite": all(finite_grads),
    "predictions": predictions,
    "references": val_texts,
    "technical_cer_not_for_research": tech_cer,
    "frozen_test_used": False,
    "warning": "ENGINEERING SMOKE ONLY. Not a research result.",
}
smoke_path = PATHS["results"] / "notebook02_smoke_test.json"
smoke_path.write_text(json.dumps(smoke_report, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {smoke_path}")
print(f"losses: {losses}")
print("WARNING: smoke CER/preds are NOT research results.")


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at hf-internal-testing/tiny-random-wav2vec2 and are newly initialized: ['wav2vec2.feature_extractor.conv_layers.1.layer_norm.bias', 'wav2vec2.feature_extractor.conv_layers.1.layer_norm.weight', 'wav2vec2.feature_extractor.conv_layers.2.layer_norm.bias', 'wav2vec2.feature_extractor.conv_layers.2.layer_norm.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at hf-internal-testing/tiny-random-wav2vec2 and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([32]) in the checkpoint and torch.Size([140]) in the model instantiated
- lm_head.weight: found shape torch.Size([32, 16]) in the checkpoint and torch.Size([140, 16]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it f

✓ Smoke samples verified from clean splits
Loading smoke model: hf-internal-testing/tiny-random-wav2vec2 @ revision=9123c4c809823cc53466e9868a1cf1c476be2e54
Requested revision: 9123c4c809823cc53466e9868a1cf1c476be2e54
Loaded revision: 9123c4c809823cc53466e9868a1cf1c476be2e54


/Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/.venv/lib/python3.9/site-packages/torch/nn/functional.py:3075: UserWarning: The operator 'aten::_ctc_loss' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:15.)
  return torch.ctc_loss(


logits shape: (4, 638, 140), vocab: 140
Wrote /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/results/notebook02_smoke_test.json
losses: [91.65789794921875, 91.6558837890625, 91.58009338378906]


## Step 6 — Final gates

Chỉ khi mọi check bắt buộc pass mới được sang Notebook 03 (full ASR trên GPU).


In [9]:
# Cell 9 — Final checks + summary (SUCCESS / BLOCKED_DATA_REVIEW / FAILED)

# NOTE: final_checks.csv is NOT in this list because it's created IN THIS CELL
# and can't be verified for current run_id before it's written
required_csv_reports = [
    PATHS["audit"] / "notebook02_manifest_checks.csv",
    PATHS["audit"] / "notebook02_train_character_frequency.csv",
    PATHS["audit"] / "notebook02_validation_oov_characters.csv",
    PATHS["audit"] / "notebook02_validation_oov_rows.csv",
    PATHS["audit"] / "notebook02_train_contamination_characters.csv",
    PATHS["audit"] / "notebook02_train_contamination_rows.csv",
    PATHS["audit"] / "notebook02_validation_contamination_characters.csv",
    PATHS["audit"] / "notebook02_validation_contamination_rows.csv",
    PATHS["audit"] / "notebook02_train_contamination_exclusions.csv",
    PATHS["audit"] / "notebook02_validation_contamination_exclusions.csv",
    PATHS["audit"] / "notebook02_text_checks.csv",
    PATHS["audit"] / "notebook02_audio_candidates.csv",
    PATHS["audit"] / "notebook02_train_audio_qa.csv",
    PATHS["audit"] / "notebook02_validation_audio_qa.csv",
    PATHS["audit"] / "notebook02_audio_qa_summary.csv",
    PATHS["audit"] / "notebook02_audio_sample_distribution.csv",
]

required_json_reports = [
    PATHS["audit"] / "notebook02_frozen_test_seal.json",
    PATHS["results"] / "notebook02_smoke_test.json",
    PATHS["results"] / "notebook02_pytest.json",
    PATHS["results"] / "notebook02_summary.json",
    PATHS["results"] / "notebook02_clean_split_contract.json",  # NEW
]

# Load reports
smoke = json.loads((PATHS["results"] / "notebook02_smoke_test.json").read_text(encoding="utf-8"))
audio_sum = pd.read_csv(PATHS["audit"] / "notebook02_audio_qa_summary.csv")
text_chk = pd.read_csv(PATHS["audit"] / "notebook02_text_checks.csv")
man_chk = pd.read_csv(PATHS["audit"] / "notebook02_manifest_checks.csv")
frozen_seal = json.loads((PATHS["audit"] / "notebook02_frozen_test_seal.json").read_text(encoding="utf-8"))
pytest_report = json.loads((PATHS["results"] / "notebook02_pytest.json").read_text(encoding="utf-8"))

# Verify CSV reports belong to current run
csv_reports_current = all(
    verify_report_metadata(p, expected_run_id=RUN_ID, expected_revision=EXPECTED_DATASET_REVISION)
    for p in required_csv_reports if p.exists()
)

# Verify JSON reports belong to current run
frozen_seal_current = verify_json_metadata(
    PATHS["audit"] / "notebook02_frozen_test_seal.json",
    expected_run_id=RUN_ID,
    expected_revision=EXPECTED_DATASET_REVISION,
    expected_processing_version=AUDIO_PROCESSING_VERSION,
)

smoke_report_current = verify_json_metadata(
    PATHS["results"] / "notebook02_smoke_test.json",
    expected_run_id=RUN_ID,
    expected_revision=EXPECTED_DATASET_REVISION,
    expected_processing_version=AUDIO_PROCESSING_VERSION,
)

pytest_report_current = verify_json_metadata(
    PATHS["results"] / "notebook02_pytest.json",
    expected_run_id=RUN_ID,
    expected_revision=EXPECTED_DATASET_REVISION,
    expected_processing_version=AUDIO_PROCESSING_VERSION,
)

all_reports_current = csv_reports_current and frozen_seal_current and smoke_report_current and pytest_report_current

# Strict cache validation
def _strict_cache_ok(frame: pd.DataFrame) -> bool:
    mask = frame["qa_hard_ok"].fillna(False).map(parse_bool)
    for _, r in frame.loc[mask].iterrows():
        v = validate_cache_wav(PROJECT_ROOT / r["cache_path"], target_sr=16_000, recompute_sha=True)
        if not (v["cache_hard_ok"] and int(v["cache_sr"]) == 16000 and int(v["cache_channels"]) == 1 
                and v["cache_waveform_finite"] and is_valid_sha256(v["cache_sha256"])):
            return False
    return True

# Check source/cache separation
source_cache_separated = all(c in qa_df.columns for c in ["source_hard_ok", "cache_hard_ok", "source_rms", "cache_rms"])

# Smoke model revision check
smoke_revision_match = (
    smoke.get("smoke_model_revision_loaded") is not None
    and smoke.get("smoke_model_revision_loaded") == SMOKE_MODEL_REVISION
)

# Cache stats verification
cache_stats_valid = (
    cache_verify["passed"]
    and cache_verify["global_fresh"] + cache_verify["global_reuse"] == len(qa_df)
)

# Clean splits verification
train_clean_no_contam = train_clean_verify["passed"]
val_clean_no_contam = val_clean_verify["passed"]
clean_vocab_no_unexpected = len(unexpected_in_vocab) == 0

# Contamination is "remediated" if clean splits have no unexpected scripts
contamination_remediated = (
    train_clean_no_contam
    and val_clean_no_contam
    and clean_vocab_no_unexpected
)

final_checks = [
    # Manifest and dataset
    {"check": "manifest_contract_pass", "passed": bool(man_chk["passed"].map(parse_bool).all())},
    {"check": "dataset_sha_stable", "passed": HUB_SHA_BEFORE == HUB_SHA_AFTER_QA == EXPECTED_DATASET_REVISION},
    
    # Frozen test isolation
    {"check": "frozen_test_restricted_load", "passed": frozen_seal.get("passed", False)},
    {"check": "frozen_test_no_forbidden_columns", "passed": len(frozen_seal.get("forbidden_columns_loaded", [])) == 0},
    
    # Text identity
    {"check": "sampled_text_identity_100pct", "passed": SAMPLED_TEXT_IDENTITY_OK},
    
    # QA sample counts
    {"check": "planned_train_equals_selected", "passed": len(qa_train) == PLANNED_QA_TRAIN},
    {"check": "planned_val_equals_selected", "passed": len(qa_val) == PLANNED_QA_VALIDATION},
    {"check": "selected_equals_qa", "passed": len(qa_df) == len(qa_selected)},
    {"check": "no_duplicate_record_uid", "passed": qa_df["record_uid"].nunique() == len(qa_df)},
    
    # Cache QA
    {"check": "source_cache_qa_separated", "passed": source_cache_separated},
    {"check": "cache_16k_mono_finite_sha", "passed": _strict_cache_ok(qa_df)},
    {"check": "audio_hard_ok_rate", "passed": bool((audio_sum["hard_ok_rate"].astype(float) >= HARD_OK_RATE_MIN).all())},
    {"check": "cache_stats_per_split_valid", "passed": cache_stats_valid},  # NEW
    
    # Contamination audits done
    {"check": "train_contamination_audit_done", "passed": (PATHS["audit"] / "notebook02_train_contamination_characters.csv").exists()},
    {"check": "validation_contamination_audit_done", "passed": (PATHS["audit"] / "notebook02_validation_contamination_characters.csv").exists()},  # NEW
    
    # Exclusion lists created
    {"check": "train_exclusion_list_created", "passed": (PATHS["audit"] / "notebook02_train_contamination_exclusions.csv").exists()},  # NEW
    {"check": "validation_exclusion_list_created", "passed": (PATHS["audit"] / "notebook02_validation_contamination_exclusions.csv").exists()},  # NEW
    
    # Clean splits verified
    {"check": "train_clean_no_contamination", "passed": train_clean_no_contam},
    {"check": "validation_clean_no_contamination", "passed": val_clean_no_contam},
    {"check": "clean_split_contract_verified", "passed": clean_split_contract_verification["passed"]},
    
    # Candidates/QA/Smoke from clean splits (asserted in earlier cells)
    {"check": "candidates_from_clean_splits", "passed": candidate_train_uids.issubset(train_clean_uids) and candidate_val_uids.issubset(validation_clean_uids)},
    {"check": "qa_samples_from_clean_splits", "passed": qa_train_uids.issubset(train_clean_uids) and qa_val_uids.issubset(validation_clean_uids)},
    {"check": "smoke_samples_from_clean_splits", "passed": smoke_train_uids.issubset(train_clean_uids) and smoke_val_uids.issubset(validation_clean_uids)},
    
    # Tokenizer from clean train - REAL VERIFICATION via provenance
    {"check": "tokenizer_provenance_verified", "passed": tokenizer_provenance_verification["passed"]},
    {"check": "tokenizer_vocab_exact_match", "passed": tok_verification.get("vocab_exact_match", False)},
    {"check": "tokenizer_vocab_no_unexpected_scripts", "passed": tok_verification.get("vocab_no_unexpected_scripts", False)},
    {"check": "tokenizer_size_correct", "passed": len(tokenizer) == len(vocab)},
    {"check": "tokenizer_no_bos_eos", "passed": tokenizer.bos_token is None and tokenizer.eos_token is None},
    {"check": "tokenizer_no_stale_tokens", "passed": "<s>" not in tokenizer.get_vocab() and "</s>" not in tokenizer.get_vocab()},
    
    # Contamination remediation (replaces manual approval)
    {"check": "contamination_remediated", "passed": contamination_remediated},  # NEW - auto-approved via exclusion
    
    # Smoke test
    {"check": "smoke_model_id_match", "passed": smoke.get("smoke_model_id") == SMOKE_MODEL_ID},
    {"check": "smoke_model_revision_match", "passed": smoke_revision_match},
    {"check": "smoke_loss_finite", "passed": bool(smoke.get("loss_finite"))},
    {"check": "smoke_logits_finite", "passed": smoke.get("logits_finite") is True},
    {"check": "smoke_gradients_finite", "passed": bool(smoke.get("gradients_finite"))},
    {"check": "smoke_optimizer_steps_2", "passed": int(smoke.get("optimizer_steps", 0)) == 2},
    {"check": "smoke_decoding_ok", "passed": isinstance(smoke.get("predictions"), list) and len(smoke["predictions"]) == SMOKE_VALIDATION_EXAMPLES},
    {"check": "smoke_no_frozen_test", "passed": smoke.get("frozen_test_used") is False},
    
    # Tests and reports
    {"check": "pytest_passed", "passed": pytest_report.get("pytest_passed", False)},
    {"check": "required_csv_reports_exist", "passed": all(p.exists() for p in required_csv_reports if "final_checks" not in str(p) and "summary" not in str(p))},
    {"check": "required_json_reports_exist", "passed": all(p.exists() for p in required_json_reports if "summary" not in str(p))},
    {"check": "frozen_seal_current_run", "passed": frozen_seal_current},
    {"check": "smoke_report_current_run", "passed": smoke_report_current},
    {"check": "all_reports_current_run", "passed": all_reports_current},
]
final_df = pd.DataFrame(final_checks)
write_dataframe_csv(final_df, PATHS["audit"] / "notebook02_final_checks.csv", metadata=RUN_META)
display(final_df)

# Determine status
all_gates_pass = bool(final_df["passed"].map(parse_bool).all())

if all_gates_pass:
    FINAL_STATUS = "SUCCESS"
    ready_for_notebook_03 = True
else:
    failed_gates = final_df.loc[~final_df["passed"].map(parse_bool), "check"].tolist()
    # Check if only contamination_remediated failed
    if failed_gates == ["contamination_remediated"]:
        FINAL_STATUS = "BLOCKED_DATA_REVIEW"
        ready_for_notebook_03 = False
    else:
        FINAL_STATUS = "FAILED"
        ready_for_notebook_03 = False

summary = {
    **RUN_META,
    "notebook": "02_asr_data_preflight_and_smoke_test",
    "status": FINAL_STATUS,
    "device": DEVICE,
    "vocab_size": len(vocab),
    
    # QA stats
    "qa_targets": {"train": QA_TRAIN_TARGET, "validation": QA_VALIDATION_TARGET},
    "qa_unique_candidates": {"train": unique_train, "validation": unique_val},
    "qa_planned": {"train": PLANNED_QA_TRAIN, "validation": PLANNED_QA_VALIDATION},
    "qa_selected": {"train": len(qa_train), "validation": len(qa_val)},
    "qa_hard_ok_rates": audio_sum.to_dict(orient="records"),
    
    # Cache stats per split (FIXED)
    "cache_stats": {
        "train": {"fresh": train_fresh, "reuse": train_reuse, "total": len(qa_train)},
        "validation": {"fresh": val_fresh, "reuse": val_reuse, "total": len(qa_val)},
        "global": {"fresh": global_fresh, "reuse": global_reuse, "total": len(qa_df)},
    },
    
    # Text identity
    "sampled_text_identity_ok": SAMPLED_TEXT_IDENTITY_OK,
    "sampled_text_identity_rate": SAMPLED_TEXT_IDENTITY_RATE,
    
    # Contamination and clean splits (NEW)
    "contamination_summary": CONTAMINATION_SUMMARY,
    "clean_splits": {
        "train_original": len(train_df),
        "train_exclusions": len(train_exclusion_df),
        "train_clean": len(train_clean_df),
        "train_clean_ordered_hash": TRAIN_CLEAN_ORDERED_HASH,
        "train_clean_set_hash": TRAIN_CLEAN_SET_HASH,
        "validation_original": len(val_df),
        "validation_exclusions": len(val_exclusion_df),
        "validation_clean": len(val_clean_df),
        "validation_clean_ordered_hash": VAL_CLEAN_ORDERED_HASH,
        "validation_clean_set_hash": VAL_CLEAN_SET_HASH,
    },
    
    # Clean split contract verification
    "clean_split_contract_path": str(PATHS["results"] / "notebook02_clean_split_contract.json"),
    "clean_split_contract_verification": clean_split_contract_verification,
    
    # Tokenizer provenance verification
    "tokenizer_provenance_verification": tokenizer_provenance_verification,
    
    # Frozen test seal
    "frozen_test_seal": {
        "allowed_columns": frozen_seal.get("allowed_columns"),
        "loaded_columns": frozen_seal.get("loaded_columns"),
        "forbidden_columns_in_file": frozen_seal.get("forbidden_columns_in_file"),
        "forbidden_columns_loaded": frozen_seal.get("forbidden_columns_loaded"),
        "passed": frozen_seal.get("passed"),
    },
    
    # Smoke test
    "smoke_model_id": SMOKE_MODEL_ID,
    "smoke_model_revision_requested": SMOKE_MODEL_REVISION,
    "smoke_model_revision_loaded": smoke.get("smoke_model_revision_loaded"),
    
    # Tests
    "pytest_passed": pytest_report.get("pytest_passed", False),
    "ready_for_notebook_03": ready_for_notebook_03,
    
    "notes": [
        f"Sampled QA covers {len(qa_df)} records, NOT full {EXPECTED_COUNTS['train']:,} train records",
        "Validation is for checkpoint/HP selection in Notebook 03",
        "Frozen test unused for vocab/model/smoke eval",
        "Smoke metrics are engineering-only",
        "source_* QA is pre-preprocess; cache_* QA is 16kHz mono WAV",
        "Contamination handled via exclusion lists, not manual approval",
        "Tokenizer built from clean train only (no unexpected scripts)",
    ],
}
(PATHS["results"] / "notebook02_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("="*60)
print(f"NOTEBOOK 02 STATUS: {FINAL_STATUS}")
print(f"ready_for_notebook_03: {ready_for_notebook_03}")
print(f"Pytest: {'PASS' if pytest_report.get('pytest_passed') else 'FAIL'}")
print(f"Cache stats (train): fresh={train_fresh} reused={train_reuse}")
print(f"Cache stats (validation): fresh={val_fresh} reused={val_reuse}")
print(f"Clean splits: train={len(train_clean_df)} validation={len(val_clean_df)}")
print("="*60)

if FINAL_STATUS == "FAILED":
    failed = final_df.loc[~final_df["passed"].map(parse_bool), "check"].tolist()
    print(f"\nFailed gates: {failed}")
    raise RuntimeError(f"Notebook 02 FAILED: {failed}")
elif FINAL_STATUS == "BLOCKED_DATA_REVIEW":
    print(f"\nBLOCKED: Clean splits still have unexpected_script contamination.")
    print(f"Train clean verification: {train_clean_verify}")
    print(f"Validation clean verification: {val_clean_verify}")
    raise RuntimeError("Notebook 02 BLOCKED — clean splits have residual contamination")
else:
    print(f"\nSUCCESS — Ready for Notebook 03")
    print(f"Train clean: {len(train_clean_df)} records (excluded {len(train_exclusion_df)})")
    print(f"Validation clean: {len(val_clean_df)} records (excluded {len(val_exclusion_df)})")
    print(f"Clean vocab size: {len(vocab)} (no unexpected scripts)")


,check,passed
0,manifest_contract_pass,True
1,dataset_sha_stable,True
2,frozen_test_restricted_load,True
3,frozen_test_no_forbidden_columns,True
4,sampled_text_identity_100pct,True
5,planned_train_equals_selected,True
6,planned_val_equals_selected,True
7,selected_equals_qa,True
8,no_duplicate_record_uid,True
9,source_cache_qa_separated,True


NOTEBOOK 02 STATUS: SUCCESS
ready_for_notebook_03: True
Pytest: PASS
Cache stats (train): fresh=0 reused=500
Cache stats (validation): fresh=0 reused=70
Clean splits: train=102507 validation=11113

SUCCESS — Ready for Notebook 03
Train clean: 102507 records (excluded 191)
Validation clean: 11113 records (excluded 19)
Clean vocab size: 140 (no unexpected scripts)


## Hand-off sang Notebook 03

Notebook 03 sẽ fine-tune ASR trên GPU với vocab/processor đã khóa và manifests RQ1 không đổi.

Nhắc lại: smoke result ở đây **không** phải kết quả nghiên cứu.
